# W11897853 baseline replication

The three regression cells estimate the selected Table 3 Panel A, Table 3 Panel B, and Table 4 Panel B scopes. Each cell runs the 21 annual OLS cross-sections using the local Fama–MacBeth package with Newey–West lag 5 and adds the mean annual adjusted R² from native annual fits. The coefficient difference is estimated through an equivalent reparameterization because the installed lagged `xtfmb` covariance contains only marginal HAC variances. Separate observation descriptives report the fit-sample variables; separate coefficient-distribution tables summarize the 21 in-memory annual fits using `asreg, by(fyear)`.


In [1]:
version 18
set more off
adopath ++ "../../ado"
use "data/main/reg.dta", clear

global y "y_bve"
global x "xrd_bve_w"
global z "capx_bve_w"
global ad "xad_bve_w"
global size "size_bve_w"
global lev "lev_bve_w"
global sampleflag "fit_bve"
global impflag "imp_bve"

count if $sampleflag==1 & $impflag==0
local observed_outcomes = r(N)
count if $sampleflag==1 & $impflag==1
local imputed_outcomes = r(N)
local imputed_pct = 100*`imputed_outcomes'/(`imputed_outcomes'+`observed_outcomes')
display "Outcome coverage: `observed_outcomes' observed; `imputed_outcomes' imputed (`imputed_pct' percent)"

egen long firm_id = group(gvkey)
xtset firm_id fyear
label variable y_bve "SD of future earnings"
label variable xrd_bve_w "R&D"
label variable capx_bve_w "Capital expenditures"
label variable xad_bve_w "Advertising"
label variable size_bve_w "Log market value (USD billions)"
label variable lev_bve_w "Leverage"

gen double rdplus = xrd_bve_w + capx_bve_w if $sampleflag==1
local adj_sum = 0
local rank_years = 0
forvalues yr = 1972/1992 {
    regress $y $x $z $ad $size $lev if $sampleflag==1 & fyear==`yr'
    if e(df_m)==5 & e(df_r)>0 {
        local adj_sum = `adj_sum' + e(r2_a)
        local rank_years = `rank_years' + 1
    }
}
local mean_adj = .
if `rank_years'>0 local mean_adj = `adj_sum'/`rank_years'
display "Full-rank annual fits: `rank_years' of 21; mean annual adjusted R-squared: `mean_adj'"

est clear
xtfmb $y $x $z $ad $size $lev if $sampleflag==1, lag(5)
est store m_base
local annual_fits = e(N_g)
xtfmb $y $x rdplus $ad $size $lev if $sampleflag==1, lag(5)
local gap_beta = _b[$x]
local gap_se = _se[$x]
local gap_t = `gap_beta'/`gap_se'
local gap_df = e(df_r)
local gap_p = 2*ttail(`gap_df',abs(`gap_t'))
est restore m_base
estadd scalar r2_a = `mean_adj'
estadd scalar annual_fits = `annual_fits'
estadd scalar full_rank_years = `rank_years'
estadd scalar observed_outcomes = `observed_outcomes'
estadd scalar imputed_outcomes = `imputed_outcomes'
estadd scalar imputed_pct = `imputed_pct'
estadd scalar rd_capx_gap = `gap_beta'
estadd scalar rd_capx_gap_se = `gap_se'
estadd scalar rd_capx_gap_t = `gap_t'
estadd scalar rd_capx_gap_p = `gap_p'

esttab m_base using "tables/table3_panel_a.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rd_capx_gap rd_capx_gap_se rd_capx_gap_t rd_capx_gap_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "R&D minus CapEx" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "R&D minus CapEx is estimated as the R&D coefficient in the equivalent R&D and R&D+CapEx parameterization; its reported p-value is two-sided.")
esttab m_base using "tables/table3_panel_a.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rd_capx_gap rd_capx_gap_se rd_capx_gap_t rd_capx_gap_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "R&D minus CapEx" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "R&D minus CapEx is estimated as the R&D coefficient in the equivalent R&D and R&D+CapEx parameterization; its reported p-value is two-sided.")
esttab m_base using "tables/table3_panel_a.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rd_capx_gap rd_capx_gap_se rd_capx_gap_t rd_capx_gap_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "R&D minus CapEx" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "R&D minus CapEx is estimated as the R&D coefficient in the equivalent R&D and R&D+CapEx parameterization; its reported p-value is two-sided.")
esttab m_base using "tables/table3_panel_a.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rd_capx_gap rd_capx_gap_se rd_capx_gap_t rd_capx_gap_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "R&D minus CapEx" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "R&D minus CapEx is estimated as the R&D coefficient in the equivalent R&D and R&D+CapEx parameterization; its reported p-value is two-sided.")

est clear
asreg $y $x $z $ad $size $lev if $sampleflag==1, by(fyear)
egen byte coefyear = tag(fyear) if $sampleflag==1
label variable _b_xrd_bve_w "R&D"
label variable _b_capx_bve_w "Capital expenditures"
label variable _b_xad_bve_w "Advertising"
label variable _b_size_bve_w "Log market value (USD billions)"
label variable _b_lev_bve_w "Leverage"
label variable _b_cons "Constant"
label variable _adjR2 "Adjusted R-squared"
estpost tabstat _b_xrd_bve_w _b_capx_bve_w _b_xad_bve_w _b_size_bve_w _b_lev_bve_w _b_cons _adjR2 if coefyear==1, statistics(N mean sd min p25 p50 p75 max) columns(statistics)
esttab using "tables/table3_panel_a_annual_coef_stats.md", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
esttab using "tables/table3_panel_a_annual_coef_stats.rtf", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
esttab using "tables/table3_panel_a_annual_coef_stats.csv", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
esttab using "tables/table3_panel_a_annual_coef_stats.tex", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

In [1]:
version 18
set more off
adopath ++ "../../ado"
use "data/main/reg.dta", clear

global y "y_price"
global x "xrd_price_w"
global z "capx_price_w"
global ad "xad_price_w"
global size "size_price_w"
global lev "lev_price_w"
global sampleflag "fit_price"
global impflag "imp_price"

count if $sampleflag==1 & $impflag==0
local observed_outcomes = r(N)
count if $sampleflag==1 & $impflag==1
local imputed_outcomes = r(N)
local imputed_pct = 100*`imputed_outcomes'/(`imputed_outcomes'+`observed_outcomes')
display "Outcome coverage: `observed_outcomes' observed; `imputed_outcomes' imputed (`imputed_pct' percent)"

egen long firm_id = group(gvkey)
xtset firm_id fyear
label variable y_price "SD of future earnings"
label variable xrd_price_w "R&D"
label variable capx_price_w "Capital expenditures"
label variable xad_price_w "Advertising"
label variable size_price_w "Log market value (USD billions)"
label variable lev_price_w "Leverage"

gen double rdplus = xrd_price_w + capx_price_w if $sampleflag==1
local adj_sum = 0
local rank_years = 0
forvalues yr = 1972/1992 {
    regress $y $x $z $ad $size $lev if $sampleflag==1 & fyear==`yr'
    if e(df_m)==5 & e(df_r)>0 {
        local adj_sum = `adj_sum' + e(r2_a)
        local rank_years = `rank_years' + 1
    }
}
local mean_adj = .
if `rank_years'>0 local mean_adj = `adj_sum'/`rank_years'
display "Full-rank annual fits: `rank_years' of 21; mean annual adjusted R-squared: `mean_adj'"

est clear
xtfmb $y $x $z $ad $size $lev if $sampleflag==1, lag(5)
est store m_base
local annual_fits = e(N_g)
xtfmb $y $x rdplus $ad $size $lev if $sampleflag==1, lag(5)
local gap_beta = _b[$x]
local gap_se = _se[$x]
local gap_t = `gap_beta'/`gap_se'
local gap_df = e(df_r)
local gap_p = 2*ttail(`gap_df',abs(`gap_t'))
est restore m_base
estadd scalar r2_a = `mean_adj'
estadd scalar annual_fits = `annual_fits'
estadd scalar full_rank_years = `rank_years'
estadd scalar observed_outcomes = `observed_outcomes'
estadd scalar imputed_outcomes = `imputed_outcomes'
estadd scalar imputed_pct = `imputed_pct'
estadd scalar rd_capx_gap = `gap_beta'
estadd scalar rd_capx_gap_se = `gap_se'
estadd scalar rd_capx_gap_t = `gap_t'
estadd scalar rd_capx_gap_p = `gap_p'

esttab m_base using "tables/table3_panel_b.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rd_capx_gap rd_capx_gap_se rd_capx_gap_t rd_capx_gap_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "R&D minus CapEx" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "R&D minus CapEx is estimated as the R&D coefficient in the equivalent R&D and R&D+CapEx parameterization; its reported p-value is two-sided.")
esttab m_base using "tables/table3_panel_b.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rd_capx_gap rd_capx_gap_se rd_capx_gap_t rd_capx_gap_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "R&D minus CapEx" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "R&D minus CapEx is estimated as the R&D coefficient in the equivalent R&D and R&D+CapEx parameterization; its reported p-value is two-sided.")
esttab m_base using "tables/table3_panel_b.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rd_capx_gap rd_capx_gap_se rd_capx_gap_t rd_capx_gap_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "R&D minus CapEx" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "R&D minus CapEx is estimated as the R&D coefficient in the equivalent R&D and R&D+CapEx parameterization; its reported p-value is two-sided.")
esttab m_base using "tables/table3_panel_b.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rd_capx_gap rd_capx_gap_se rd_capx_gap_t rd_capx_gap_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "R&D minus CapEx" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "R&D minus CapEx is estimated as the R&D coefficient in the equivalent R&D and R&D+CapEx parameterization; its reported p-value is two-sided.")

est clear
asreg $y $x $z $ad $size $lev if $sampleflag==1, by(fyear)
egen byte coefyear = tag(fyear) if $sampleflag==1
label variable _b_xrd_price_w "R&D"
label variable _b_capx_price_w "Capital expenditures"
label variable _b_xad_price_w "Advertising"
label variable _b_size_price_w "Log market value (USD billions)"
label variable _b_lev_price_w "Leverage"
label variable _b_cons "Constant"
label variable _adjR2 "Adjusted R-squared"
estpost tabstat _b_xrd_price_w _b_capx_price_w _b_xad_price_w _b_size_price_w _b_lev_price_w _b_cons _adjR2 if coefyear==1, statistics(N mean sd min p25 p50 p75 max) columns(statistics)
esttab using "tables/table3_panel_b_annual_coef_stats.md", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
esttab using "tables/table3_panel_b_annual_coef_stats.rtf", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
esttab using "tables/table3_panel_b_annual_coef_stats.csv", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
esttab using "tables/table3_panel_b_annual_coef_stats.tex", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

In [1]:
version 18
set more off
adopath ++ "../../ado"
use "data/main/reg.dta", clear

global y "y_cap"
global x "caprd_bve_w"
global z "ppent_bve_w"
global ad "xad_cap_w"
global size "size_cap_w"
global lev "lev_cap_w"
global sampleflag "fit_cap"
global impflag "imp_cap"

count if $sampleflag==1 & $impflag==0
local observed_outcomes = r(N)
count if $sampleflag==1 & $impflag==1
local imputed_outcomes = r(N)
local imputed_pct = 100*`imputed_outcomes'/(`imputed_outcomes'+`observed_outcomes')
display "Outcome coverage: `observed_outcomes' observed; `imputed_outcomes' imputed (`imputed_pct' percent)"

egen long firm_id = group(gvkey)
xtset firm_id fyear
label variable y_cap "SD of gross future earnings"
label variable caprd_bve_w "Capitalized R&D"
label variable ppent_bve_w "Net PPE"
label variable xad_cap_w "Advertising"
label variable size_cap_w "Log market value (USD billions)"
label variable lev_cap_w "Leverage"

gen double rdplus = caprd_bve_w + ppent_bve_w if $sampleflag==1
local adj_sum = 0
local rank_years = 0
forvalues yr = 1972/1992 {
    regress $y $x $z $ad $size $lev if $sampleflag==1 & fyear==`yr'
    if e(df_m)==5 & e(df_r)>0 {
        local adj_sum = `adj_sum' + e(r2_a)
        local rank_years = `rank_years' + 1
    }
}
local mean_adj = .
if `rank_years'>0 local mean_adj = `adj_sum'/`rank_years'
display "Full-rank annual fits: `rank_years' of 21; mean annual adjusted R-squared: `mean_adj'"

est clear
xtfmb $y $x $z $ad $size $lev if $sampleflag==1, lag(5)
est store m_base
local annual_fits = e(N_g)
xtfmb $y $x rdplus $ad $size $lev if $sampleflag==1, lag(5)
local gap_beta = _b[$x]
local gap_se = _se[$x]
local gap_t = `gap_beta'/`gap_se'
local gap_df = e(df_r)
local gap_p = 2*ttail(`gap_df',abs(`gap_t'))
est restore m_base
estadd scalar r2_a = `mean_adj'
estadd scalar annual_fits = `annual_fits'
estadd scalar full_rank_years = `rank_years'
estadd scalar observed_outcomes = `observed_outcomes'
estadd scalar imputed_outcomes = `imputed_outcomes'
estadd scalar imputed_pct = `imputed_pct'
estadd scalar rdcap_ppe_gap = `gap_beta'
estadd scalar rdcap_ppe_se = `gap_se'
estadd scalar rdcap_ppe_t = `gap_t'
estadd scalar rdcap_ppe_p = `gap_p'

esttab m_base using "tables/table4_panel_b.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rdcap_ppe_gap rdcap_ppe_se rdcap_ppe_t rdcap_ppe_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "Capitalized R&D minus PPE" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "Capitalized R&D minus PPE is estimated as the R&D coefficient in the equivalent CAPR&D and CAPR&D+PPE parameterization; its reported p-value is two-sided.")
esttab m_base using "tables/table4_panel_b.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rdcap_ppe_gap rdcap_ppe_se rdcap_ppe_t rdcap_ppe_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "Capitalized R&D minus PPE" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "Capitalized R&D minus PPE is estimated as the R&D coefficient in the equivalent CAPR&D and CAPR&D+PPE parameterization; its reported p-value is two-sided.")
esttab m_base using "tables/table4_panel_b.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rdcap_ppe_gap rdcap_ppe_se rdcap_ppe_t rdcap_ppe_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "Capitalized R&D minus PPE" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "Capitalized R&D minus PPE is estimated as the R&D coefficient in the equivalent CAPR&D and CAPR&D+PPE parameterization; its reported p-value is two-sided.")
esttab m_base using "tables/table4_panel_b.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label stats(N r2 r2_a annual_fits full_rank_years observed_outcomes imputed_outcomes imputed_pct rdcap_ppe_gap rdcap_ppe_se rdcap_ppe_t rdcap_ppe_p, fmt(0 3 3 0 0 0 0 1 3 3 2 3) labels("Observations" "Average R-squared" "Mean annual adjusted R-squared" "Annual fits" "Full-rank years" "Observed outcomes" "Imputed outcomes" "Imputed percent" "Capitalized R&D minus PPE" "Difference NW SE" "Difference NW t" "Difference two-sided p")) addnotes("Separate annual OLS cross-sections, averaged over fiscal years 1972–1992; coefficient-series inference uses Newey–West lag 5." "Capitalized R&D minus PPE is estimated as the R&D coefficient in the equivalent CAPR&D and CAPR&D+PPE parameterization; its reported p-value is two-sided.")

est clear
asreg $y $x $z $ad $size $lev if $sampleflag==1, by(fyear)
egen byte coefyear = tag(fyear) if $sampleflag==1
label variable _b_caprd_bve_w "Capitalized R&D"
label variable _b_ppent_bve_w "Net PPE"
label variable _b_xad_cap_w "Advertising"
label variable _b_size_cap_w "Log market value (USD billions)"
label variable _b_lev_cap_w "Leverage"
label variable _b_cons "Constant"
label variable _adjR2 "Adjusted R-squared"
estpost tabstat _b_caprd_bve_w _b_ppent_bve_w _b_xad_cap_w _b_size_cap_w _b_lev_cap_w _b_cons _adjR2 if coefyear==1, statistics(N mean sd min p25 p50 p75 max) columns(statistics)
esttab using "tables/table4_panel_b_annual_coef_stats.md", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
esttab using "tables/table4_panel_b_annual_coef_stats.rtf", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
esttab using "tables/table4_panel_b_annual_coef_stats.csv", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
esttab using "tables/table4_panel_b_annual_coef_stats.tex", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) min(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f)) max(fmt(%9.3f))") label
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

In [1]:
version 18
set more off
adopath ++ "../../ado"
use "data/main/reg.dta", clear

global sampleflag "fit_bve"
label variable y_bve "SD of future earnings"
label variable xrd_bve_w "R&D"
label variable capx_bve_w "Capital expenditures"
label variable xad_bve_w "Advertising"
label variable size_bve_w "Log market value (USD billions)"
label variable lev_bve_w "Leverage"
est clear
estpost tabstat y_bve xrd_bve_w capx_bve_w xad_bve_w size_bve_w lev_bve_w if $sampleflag==1, statistics(N mean sd p25 p50 p75) columns(statistics)
esttab using "tables/sample_bve_descriptives.md", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/sample_bve_descriptives.rtf", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/sample_bve_descriptives.csv", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/sample_bve_descriptives.tex", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

In [1]:
version 18
set more off
adopath ++ "../../ado"
use "data/main/reg.dta", clear

global sampleflag "fit_price"
label variable y_price "SD of future earnings"
label variable xrd_price_w "R&D"
label variable capx_price_w "Capital expenditures"
label variable xad_price_w "Advertising"
label variable size_price_w "Log market value (USD billions)"
label variable lev_price_w "Leverage"
est clear
estpost tabstat y_price xrd_price_w capx_price_w xad_price_w size_price_w lev_price_w if $sampleflag==1, statistics(N mean sd p25 p50 p75) columns(statistics)
esttab using "tables/sample_price_descriptives.md", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/sample_price_descriptives.rtf", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/sample_price_descriptives.csv", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/sample_price_descriptives.tex", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

In [1]:
version 18
set more off
adopath ++ "../../ado"
use "data/main/reg.dta", clear

global sampleflag "fit_cap"
label variable y_cap "SD of gross future earnings"
label variable caprd_bve_w "Capitalized R&D"
label variable ppent_bve_w "Net PPE"
label variable xad_cap_w "Advertising"
label variable size_cap_w "Log market value (USD billions)"
label variable lev_cap_w "Leverage"
est clear
estpost tabstat y_cap caprd_bve_w ppent_bve_w xad_cap_w size_cap_w lev_cap_w if $sampleflag==1, statistics(N mean sd p25 p50 p75) columns(statistics)
esttab using "tables/sample_cap_descriptives.md", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/sample_cap_descriptives.rtf", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/sample_cap_descriptives.csv", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/sample_cap_descriptives.tex", replace cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do